# Load & Orchestrate — Week 3, demonstrated

**Home Safe (Kent domiciliary/personal care market intelligence) — LSA Data Engineering Internship, September 2026 cohort**
**Mentor reference notebook — Venkat, 17 Sept 2026**

This notebook covers Week 3 of `HomeSafe_Week1-4_Project_Spec`:

1. Design a **star schema** (one fact table, three dimensions)
2. **Data-quality checks as gates**: a blocking failure stops the load
3. An **idempotent load** into SQLite, so re-running never duplicates rows
4. An **audit table**, so every run (loaded, blocked or failed) leaves a record
5. **Automated tests** (pytest, no network, no keys)
6. A **plain script** a scheduler can run, with meaningful exit codes, plus cron / Task Scheduler / GitHub Actions set-ups
7. A Week 4 preview query showing the schema answers the business question

**Read the next cell before the code.** As in Weeks 1–2, it separates what was run and tested from what still needs a real run on your machine.

## What was checked before writing this, and how

**Executed and tested here (17 Sept 2026, Python 3.11, SQLite 3.45):** every code cell in this notebook ran top to bottom. The 21 pytest tests pass. The suite was also mutation-checked: when the idempotent `DELETE`, the quality gate, or the drift threshold was deliberately broken, tests failed each time, so they catch real faults. `run_pipeline.py` was run from the command line and returned exit code 0 (loaded), 2 (blocked) and 1 (missing input) as designed.

**Confirmed against official documentation:**
- SQLite UPSERT (`INSERT … ON CONFLICT(…) DO UPDATE SET col = excluded.col`) exists from SQLite **3.24.0**. The conflict target must be a `PRIMARY KEY`/`UNIQUE` constraint ([sqlite.org/lang_upsert](https://www.sqlite.org/lang_upsert.html)). Python's bundled SQLite is newer than this on any current install.
- GitHub Actions `schedule` ([GitHub docs source](https://github.com/github/docs), *events-that-trigger-workflows*): POSIX cron, **UTC by default**, **shortest interval 5 minutes**, runs **only on the default branch**, **can be delayed or dropped at high load (the start of each hour is busiest)**, and in a **public repo it auto-disables after 60 days with no repository activity**.

**Standard SQLite behaviour this notebook relies on:**
- `PRAGMA foreign_keys = ON` is off by default and must be set on **every connection**.
- `with conn:` commits on success and rolls back on an exception. The rollback test below proves it.

**Not yet confirmed. Needs a real run:**
- Everything here ran against a **labelled sample** shaped exactly like Week 1/Week 2 outputs (same column names, taken from the Week 1 `flatten_location_detail()` and Week 2 `enrich_provider()` code). Threshold choices (5% bad postcodes, 20% row-count swing) are team decisions, not facts. Tune them against real Kent data.
- The cron, Task Scheduler and GitHub Actions snippets are printed, not executed. This environment can't schedule anything on your machine.

## Environment check

In [1]:
import sys, sqlite3
print(f"Python {sys.version.split()[0]} | SQLite {sqlite3.sqlite_version}")
for mod in ["pandas", "pytest", "pyarrow"]:
    try:
        __import__(mod); print(f"OK      - {mod}")
    except ImportError:
        print(f"MISSING - {mod}  (pip install {mod})")
# !pip install pandas pytest pyarrow --quiet

Python 3.11.15 | SQLite 3.45.1


OK      - pandas
OK      - pytest
OK      - pyarrow


## Task 1 — The star schema

**First question: what is one row in the fact table?** Not "a provider". Providers own many locations, and ratings belong to locations. The grain is **one CQC location, as observed on one snapshot date**. That grain lets the same table hold September's run and October's run side by side, which is what makes "how is the market changing?" answerable in Week 4.

| Table | Grain / key | Comes from | Notes |
|---|---|---|---|
| `fact_location_snapshot` | (`snapshot_date`, `location_id`) | Week 1 locations CSV | rating, registration status, postcode validity flag |
| `dim_provider` | `provider_id` | Week 2 enriched providers | Companies House fields, `match_method`, `match_score`. Updated in place (SCD Type 1) |
| `dim_area` | `area_key` (surrogate), `local_authority` unique | derived | `is_kent_district` flags anything outside the 12 (e.g. Medway) instead of dropping it |
| `dim_rating_period` | `YYYY-MM` or `NOT_RATED` | derived from `overall_rating_date` | **Design decision:** month of the rating's report date. Never-inspected locations get `NOT_RATED`, not NULL, so the foreign key always resolves |
| `load_audit` | `run_id` | pipeline | not part of the star. Operational record of every run |

**Challenge for teams:** the spec lists a rating-period dimension. With only *current* ratings, it describes when each location was last rated. It does not give rating history. For history, the Week 1 extract would need CQC's `historicRatings[]` array. Decide as a team whether that's worth it, and write the choice in your schema design doc.

In [2]:
%%writefile load_week3.py
"""Home Safe -- Week 3: Load & Orchestrate.

Star-schema load into SQLite with:
  * idempotent re-runs (delete-then-insert per snapshot_date, inside ONE transaction),
  * data-quality checks that act as GATES (a blocking failure stops the load),
  * a load_audit table so every run -- passed or blocked -- leaves a record.

No network calls in this module. It reads Week 1 / Week 2 outputs that already
exist on disk, so the whole thing is testable with pytest and no API keys.
"""
from __future__ import annotations

import math
import re
import sqlite3
import uuid
from dataclasses import dataclass, field
from datetime import date, datetime, timezone

import pandas as pd

# ---------------------------------------------------------------------------
# Constants (same lists as Weeks 1-2 -- re-check the district list before Week 4,
# Kent local government reorganisation is still in progress)
# ---------------------------------------------------------------------------
KENT_LOCAL_AUTHORITIES = [
    "Ashford", "Canterbury", "Dartford", "Dover", "Folkestone and Hythe",
    "Gravesham", "Maidstone", "Sevenoaks", "Swale", "Thanet",
    "Tonbridge and Malling", "Tunbridge Wells",
]

# CQC's four published overall ratings. A blank rating means "not yet rated",
# which is legitimate -- anything ELSE is a data problem.
VALID_RATINGS = {"Outstanding", "Good", "Requires improvement", "Inadequate"}

UK_POSTCODE_RE = re.compile(r"^[A-Z]{1,2}\d[A-Z\d]?\s?\d[A-Z]{2}$")

NOT_RATED_KEY = "NOT_RATED"

REQUIRED_LOCATION_COLUMNS = [
    "location_id", "provider_id", "location_name", "postal_code",
    "local_authority", "registration_status", "overall_rating", "overall_rating_date",
]
REQUIRED_PROVIDER_COLUMNS = ["provider_id", "provider_name"]

# Optional provider columns carried into dim_provider if present in Week 2 output.
PROVIDER_OPTIONAL_COLUMNS = [
    "ownership_type", "companies_house_number", "charity_number", "match_method",
    "match_score", "ch_company_status", "ch_date_of_creation", "ch_company_type",
]

# Thresholds -- team decisions, not facts. Document any change in the runbook.
MAX_INVALID_POSTCODE_SHARE = 0.05   # >5% malformed postcodes blocks the load
MAX_ROW_COUNT_CHANGE_SHARE = 0.20   # >20% swing vs the previous snapshot blocks


# ---------------------------------------------------------------------------
# Small pure helpers
# ---------------------------------------------------------------------------
def is_blank(value) -> bool:
    """NaN-aware blank check. NaN is truthy in Python -- never use `if value`
    on a pandas cell (the Week 2 bug)."""
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    return str(value).strip() == "" or str(value).strip().lower() in {"nan", "nat", "none"}


def clean_text(value):
    return None if is_blank(value) else str(value).strip()


def is_valid_uk_postcode(value) -> bool:
    if is_blank(value):
        return False
    return bool(UK_POSTCODE_RE.match(str(value).strip().upper()))


def parse_iso_date(value):
    """Return a datetime.date, or None if blank/unparseable. Accepts
    '2024-05-01' and '2024-05-01T00:00:00' style strings."""
    if is_blank(value):
        return None
    try:
        return pd.to_datetime(str(value).strip()[:10], format="%Y-%m-%d").date()
    except (ValueError, TypeError):
        return None


def rating_period_key(rating, rating_date) -> str:
    """dim_rating_period grain = calendar month of the rating's report date.
    A location with no rating (never inspected) maps to NOT_RATED, not NULL,
    so the fact table's foreign key is always populated."""
    if is_blank(rating):
        return NOT_RATED_KEY
    d = parse_iso_date(rating_date)
    return NOT_RATED_KEY if d is None else f"{d.year:04d}-{d.month:02d}"


# ---------------------------------------------------------------------------
# Data-quality checks
# ---------------------------------------------------------------------------
@dataclass
class CheckResult:
    name: str
    passed: bool
    blocking: bool
    detail: str = ""


@dataclass
class QualityReport:
    results: list = field(default_factory=list)

    def add(self, name, passed, blocking=True, detail=""):
        self.results.append(CheckResult(name, bool(passed), blocking, detail))

    @property
    def blocking_failures(self):
        return [r for r in self.results if r.blocking and not r.passed]

    @property
    def warnings(self):
        return [r for r in self.results if not r.blocking and not r.passed]

    @property
    def ok_to_load(self) -> bool:
        return not self.blocking_failures

    def to_frame(self) -> pd.DataFrame:
        return pd.DataFrame([r.__dict__ for r in self.results])


def run_quality_checks(locations: pd.DataFrame, providers: pd.DataFrame,
                       snapshot_date: date, previous_row_count: int | None = None,
                       allow_row_count_change: bool = False) -> QualityReport:
    """Run every check BEFORE touching the database. Blocking failures stop the
    load; warnings are recorded in load_audit but let the load continue."""
    rep = QualityReport()

    # 1. Schema -- if columns are missing, nothing else is meaningful.
    missing_loc = [c for c in REQUIRED_LOCATION_COLUMNS if c not in locations.columns]
    missing_prov = [c for c in REQUIRED_PROVIDER_COLUMNS if c not in providers.columns]
    rep.add("required_columns_present", not missing_loc and not missing_prov,
            detail=f"missing locations={missing_loc} providers={missing_prov}")
    if missing_loc or missing_prov:
        return rep

    # 2. Not empty
    rep.add("locations_not_empty", len(locations) > 0, detail=f"{len(locations)} rows")

    # 3. Keys present
    blank_loc_ids = int(locations["location_id"].apply(is_blank).sum())
    blank_prov_ids = int(locations["provider_id"].apply(is_blank).sum())
    rep.add("location_and_provider_ids_present", blank_loc_ids == 0 and blank_prov_ids == 0,
            detail=f"blank location_id={blank_loc_ids}, blank provider_id={blank_prov_ids}")

    # 4. Uniqueness (the spec's "no duplicate IDs" check, applied at both grains)
    dup_loc = locations["location_id"][locations["location_id"].duplicated()].unique().tolist()
    rep.add("location_id_unique", not dup_loc, detail=f"duplicates: {dup_loc[:10]}")
    dup_prov = providers["provider_id"][providers["provider_id"].duplicated()].unique().tolist()
    rep.add("provider_id_unique", not dup_prov, detail=f"duplicates: {dup_prov[:10]}")

    # 5. Referential integrity -- every location's provider must exist in dim_provider
    orphans = sorted(set(locations["provider_id"].dropna()) - set(providers["provider_id"].dropna()))
    rep.add("every_location_has_a_provider", not orphans, detail=f"orphan provider_ids: {orphans[:10]}")

    # 6. Rating domain -- blank is fine (not yet rated), an unknown label is not
    bad_ratings = sorted({str(r) for r in locations["overall_rating"]
                          if not is_blank(r) and str(r).strip() not in VALID_RATINGS})
    rep.add("overall_rating_in_allowed_values", not bad_ratings, detail=f"unexpected: {bad_ratings}")

    # 7. Rating dates must parse and not be in the future
    rated = locations[~locations["overall_rating"].apply(is_blank)]
    parsed = rated["overall_rating_date"].apply(parse_iso_date)
    unparseable = int(parsed.isna().sum())
    future = int(sum(1 for d in parsed if d is not None and d > snapshot_date))
    rep.add("rating_dates_valid_and_not_future", unparseable == 0 and future == 0,
            detail=f"unparseable={unparseable}, after snapshot_date={future}")

    # 8. Postcodes -- a few bad ones is a warning; lots means something upstream broke
    n = max(len(locations), 1)
    invalid_pc = int((~locations["postal_code"].apply(is_valid_uk_postcode)).sum())
    share = invalid_pc / n
    rep.add("postcode_invalid_share_within_threshold", share <= MAX_INVALID_POSTCODE_SHARE,
            detail=f"{invalid_pc} invalid ({share:.1%}); limit {MAX_INVALID_POSTCODE_SHARE:.0%}")
    rep.add("all_postcodes_valid", invalid_pc == 0, blocking=False,
            detail=f"{invalid_pc} invalid postcode(s) loaded and flagged")

    # 9. Local authority outside the 12 districts -- flag, never drop (Week 2 rule)
    la_lookup = {la.lower() for la in KENT_LOCAL_AUTHORITIES}
    outside = sorted({str(v) for v in locations["local_authority"]
                      if is_blank(v) or str(v).strip().lower() not in la_lookup})
    rep.add("local_authority_in_kent_list", not outside, blocking=False,
            detail=f"outside list: {outside}")

    # 10. Volume drift vs the last successful snapshot
    if previous_row_count:
        change = abs(len(locations) - previous_row_count) / previous_row_count
        rep.add("row_count_change_within_threshold",
                change <= MAX_ROW_COUNT_CHANGE_SHARE or allow_row_count_change,
                detail=(f"{previous_row_count} -> {len(locations)} ({change:.0%}); "
                        f"limit {MAX_ROW_COUNT_CHANGE_SHARE:.0%}"
                        + ("; OVERRIDDEN by operator" if allow_row_count_change else "")))
    return rep


# ---------------------------------------------------------------------------
# Schema
# ---------------------------------------------------------------------------
SCHEMA_SQL = """
PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS dim_area (
    area_key          INTEGER PRIMARY KEY,
    local_authority   TEXT NOT NULL UNIQUE,
    is_kent_district  INTEGER NOT NULL CHECK (is_kent_district IN (0, 1))
);

CREATE TABLE IF NOT EXISTS dim_provider (
    provider_id            TEXT PRIMARY KEY,
    provider_name          TEXT NOT NULL,
    ownership_type         TEXT,
    companies_house_number TEXT,
    charity_number         TEXT,
    match_method           TEXT,
    match_score            REAL,
    ch_company_status      TEXT,
    ch_date_of_creation    TEXT,
    ch_company_type        TEXT,
    last_loaded_at         TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS dim_rating_period (
    rating_period_key  TEXT PRIMARY KEY,      -- 'YYYY-MM' or 'NOT_RATED'
    year               INTEGER,
    month              INTEGER,
    quarter            INTEGER
);

CREATE TABLE IF NOT EXISTS fact_location_snapshot (
    snapshot_date        TEXT NOT NULL,       -- the day this pipeline run observed CQC
    location_id          TEXT NOT NULL,
    provider_id          TEXT NOT NULL REFERENCES dim_provider(provider_id),
    area_key             INTEGER NOT NULL REFERENCES dim_area(area_key),
    rating_period_key    TEXT NOT NULL REFERENCES dim_rating_period(rating_period_key),
    location_name        TEXT,
    postal_code          TEXT,
    postcode_valid       INTEGER NOT NULL,
    registration_status  TEXT,
    overall_rating       TEXT,
    overall_rating_date  TEXT,
    is_rated             INTEGER NOT NULL,
    PRIMARY KEY (snapshot_date, location_id)
);

CREATE TABLE IF NOT EXISTS load_audit (
    run_id          TEXT PRIMARY KEY,
    snapshot_date   TEXT NOT NULL,
    started_at      TEXT NOT NULL,
    finished_at     TEXT,
    status          TEXT NOT NULL,            -- LOADED | BLOCKED | FAILED
    rows_in         INTEGER,
    rows_loaded     INTEGER,
    blocking_failed TEXT,
    warnings        TEXT
);
"""


def connect(db_path: str) -> sqlite3.Connection:
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")  # per-connection in SQLite, off by default
    return conn


def create_schema(conn: sqlite3.Connection) -> None:
    conn.executescript(SCHEMA_SQL)


def previous_snapshot_row_count(conn, snapshot_date: date):
    """Row count of the most recent LOADED snapshot strictly before this one."""
    row = conn.execute(
        """SELECT rows_loaded FROM load_audit
           WHERE status = 'LOADED' AND snapshot_date < ?
           ORDER BY snapshot_date DESC, finished_at DESC LIMIT 1""",
        (snapshot_date.isoformat(),),
    ).fetchone()
    return row[0] if row else None


# ---------------------------------------------------------------------------
# Load
# ---------------------------------------------------------------------------
def _upsert_areas(conn, local_authorities):
    kent = {la.lower(): la for la in KENT_LOCAL_AUTHORITIES}
    for raw in sorted({clean_text(v) or "UNKNOWN" for v in local_authorities}):
        canonical = kent.get(raw.lower(), raw)
        conn.execute(
            """INSERT INTO dim_area (local_authority, is_kent_district) VALUES (?, ?)
               ON CONFLICT(local_authority) DO NOTHING""",
            (canonical, int(raw.lower() in kent)),
        )
    return {name.lower(): key for key, name in
            conn.execute("SELECT area_key, local_authority FROM dim_area")}


def _upsert_providers(conn, providers, loaded_at):
    cols = ["provider_id", "provider_name"] + PROVIDER_OPTIONAL_COLUMNS + ["last_loaded_at"]
    placeholders = ", ".join("?" for _ in cols)
    updates = ", ".join(f"{c} = excluded.{c}" for c in cols if c != "provider_id")
    sql = (f"INSERT INTO dim_provider ({', '.join(cols)}) VALUES ({placeholders}) "
           f"ON CONFLICT(provider_id) DO UPDATE SET {updates}")
    for _, r in providers.iterrows():
        values = [clean_text(r["provider_id"]), clean_text(r["provider_name"])]
        for c in PROVIDER_OPTIONAL_COLUMNS:
            v = r[c] if c in providers.columns else None
            if c == "match_score":
                values.append(None if is_blank(v) else float(v))
            else:
                values.append(clean_text(v))
        values.append(loaded_at)
        conn.execute(sql, values)


def _upsert_rating_periods(conn, keys):
    for k in sorted(set(keys)):
        if k == NOT_RATED_KEY:
            row = (k, None, None, None)
        else:
            y, m = int(k[:4]), int(k[5:7])
            row = (k, y, m, (m - 1) // 3 + 1)
        conn.execute("INSERT INTO dim_rating_period VALUES (?, ?, ?, ?) "
                     "ON CONFLICT(rating_period_key) DO NOTHING", row)


def load_snapshot(conn, locations, providers, snapshot_date: date):
    """Idempotent load of ONE snapshot. Everything happens in a single
    transaction: dims upserted, this snapshot_date's fact rows deleted, then
    re-inserted. If anything raises, the whole thing rolls back -- the database
    is never left half-loaded."""
    loaded_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    snap = snapshot_date.isoformat()
    with conn:  # sqlite3: commit on success, rollback on exception
        area_keys = _upsert_areas(conn, locations["local_authority"])
        _upsert_providers(conn, providers, loaded_at)
        rp_keys = [rating_period_key(r, d) for r, d in
                   zip(locations["overall_rating"], locations["overall_rating_date"])]
        _upsert_rating_periods(conn, rp_keys)

        conn.execute("DELETE FROM fact_location_snapshot WHERE snapshot_date = ?", (snap,))
        rows = []
        for (_, r), rp in zip(locations.iterrows(), rp_keys):
            la = (clean_text(r["local_authority"]) or "UNKNOWN").lower()
            pc = clean_text(r["postal_code"])
            rating = clean_text(r["overall_rating"])
            rdate = parse_iso_date(r["overall_rating_date"])
            rows.append((
                snap, clean_text(r["location_id"]), clean_text(r["provider_id"]),
                area_keys[la], rp, clean_text(r["location_name"]),
                pc.upper() if pc else None, int(is_valid_uk_postcode(pc)),
                clean_text(r["registration_status"]), rating,
                rdate.isoformat() if rdate else None, int(rating is not None),
            ))
        conn.executemany(
            "INSERT INTO fact_location_snapshot VALUES (?,?,?,?,?,?,?,?,?,?,?,?)", rows)
        loaded = conn.execute("SELECT COUNT(*) FROM fact_location_snapshot WHERE snapshot_date = ?",
                              (snap,)).fetchone()[0]
        if loaded != len(locations):  # post-load reconciliation, still inside the transaction
            raise RuntimeError(f"Reconciliation failed: {len(locations)} in, {loaded} loaded")
    return loaded


def _write_audit(conn, run_id, snapshot_date, started_at, status, rows_in,
                 rows_loaded, report: QualityReport | None, message=""):
    blocking = "; ".join(f"{r.name}: {r.detail}" for r in report.blocking_failures) if report else ""
    warns = "; ".join(f"{r.name}: {r.detail}" for r in report.warnings) if report else ""
    if message:
        blocking = (blocking + "; " if blocking else "") + message
    with conn:
        conn.execute(
            "INSERT INTO load_audit VALUES (?,?,?,?,?,?,?,?,?)",
            (run_id, snapshot_date.isoformat(), started_at,
             datetime.now(timezone.utc).isoformat(timespec="seconds"),
             status, rows_in, rows_loaded, blocking, warns))


def run_pipeline(conn, locations, providers, snapshot_date: date,
                 allow_row_count_change: bool = False):
    """Checks -> gate -> load -> audit. Returns (status, QualityReport)."""
    create_schema(conn)
    run_id = uuid.uuid4().hex
    started = datetime.now(timezone.utc).isoformat(timespec="seconds")
    prev = previous_snapshot_row_count(conn, snapshot_date)
    report = run_quality_checks(locations, providers, snapshot_date, prev, allow_row_count_change)

    if not report.ok_to_load:
        _write_audit(conn, run_id, snapshot_date, started, "BLOCKED", len(locations), 0, report)
        return "BLOCKED", report
    try:
        loaded = load_snapshot(conn, locations, providers, snapshot_date)
    except Exception as e:  # recorded, then re-raised so a scheduler sees a failure
        _write_audit(conn, run_id, snapshot_date, started, "FAILED", len(locations), 0,
                     report, message=f"{type(e).__name__}: {e}")
        raise
    _write_audit(conn, run_id, snapshot_date, started, "LOADED", len(locations), loaded, report)
    return "LOADED", report


# ---------------------------------------------------------------------------
# A Week 4 preview query -- proves the star schema answers the business question
# ---------------------------------------------------------------------------
DENSITY_BY_AREA_SQL = """
SELECT a.local_authority,
       COUNT(DISTINCT f.provider_id)                                   AS providers,
       COUNT(*)                                                        AS locations,
       SUM(CASE WHEN f.overall_rating = 'Outstanding' THEN 1 ELSE 0 END)          AS outstanding,
       SUM(CASE WHEN f.overall_rating = 'Good' THEN 1 ELSE 0 END)                 AS good,
       SUM(CASE WHEN f.overall_rating = 'Requires improvement' THEN 1 ELSE 0 END) AS requires_improvement,
       SUM(CASE WHEN f.overall_rating = 'Inadequate' THEN 1 ELSE 0 END)           AS inadequate,
       SUM(CASE WHEN f.is_rated = 0 THEN 1 ELSE 0 END)                            AS not_yet_rated,
       -- NULL = NULL is NULL in SQL, so a bare SUM(rating = 'Good') returns NULL for a
       -- district whose only locations are unrated. CASE ... ELSE 0 avoids that.
       ROUND(1.0 * SUM(CASE WHEN f.overall_rating IN ('Outstanding','Good') THEN 1 ELSE 0 END)
             / NULLIF(SUM(f.is_rated), 0), 2)                                     AS good_or_better_share
FROM fact_location_snapshot f
JOIN dim_area a ON a.area_key = f.area_key
WHERE f.snapshot_date = (SELECT MAX(snapshot_date) FROM fact_location_snapshot)
  AND f.registration_status = 'Registered'
GROUP BY a.local_authority
ORDER BY locations DESC;
"""


Writing load_week3.py


In [3]:
%%writefile sample_data.py
"""Small, clearly-labelled SAMPLE inputs shaped exactly like the real Week 1 and
Week 2 outputs (same column names). Used when the real files aren't present and
by the tests. Numbers produced from these are illustrative only."""
import numpy as np
import pandas as pd


def sample_providers() -> pd.DataFrame:
    return pd.DataFrame([
        {"provider_id": "1-000001", "provider_name": "Home Safe Domiciliary Care Ltd",
         "ownership_type": "Organisation", "companies_house_number": "00000006",
         "charity_number": np.nan, "match_method": "direct_number", "match_score": 100.0,
         "ch_company_status": "active", "ch_date_of_creation": "2012-04-02",
         "ch_company_type": "ltd"},
        {"provider_id": "1-000002", "provider_name": "Kent Homecare Services Limited",
         "ownership_type": "Organisation", "companies_house_number": np.nan,
         "charity_number": np.nan, "match_method": "fuzzy_name", "match_score": 92.0,
         "ch_company_status": "active", "ch_date_of_creation": "2014-11-19",
         "ch_company_type": "ltd"},
        {"provider_id": "1-000003", "provider_name": "Garden of England Care Trust",
         "ownership_type": "Organisation", "companies_house_number": np.nan,
         "charity_number": "1123456", "match_method": "charity_no_ch_expected",
         "match_score": np.nan, "ch_company_status": np.nan, "ch_date_of_creation": np.nan,
         "ch_company_type": np.nan},
        {"provider_id": "1-000004", "provider_name": "A. Patel (Personal Care)",
         "ownership_type": "Individual", "companies_house_number": np.nan,
         "charity_number": np.nan, "match_method": "fuzzy_no_match_above_threshold",
         "match_score": 41.0, "ch_company_status": np.nan, "ch_date_of_creation": np.nan,
         "ch_company_type": np.nan},
    ])


def sample_locations() -> pd.DataFrame:
    rows = [
        ("1-100001", "1-000001", "Home Safe Ashford", "TN23 1AB", "Ashford", "Registered", "Good", "2024-05-14"),
        ("1-100002", "1-000001", "Home Safe Canterbury", "CT1 2AB", "Canterbury", "Registered", "Outstanding", "2023-11-02"),
        ("1-100003", "1-000002", "Kent Homecare Canterbury", "CT2 7AB", "Canterbury", "Registered", "Requires improvement", "2025-02-20"),
        ("1-100004", "1-000002", "Kent Homecare Thanet", "CT9 1AB", "Thanet", "Registered", "Good", "2025-06-30"),
        ("1-100005", "1-000003", "Garden of England Maidstone", "ME14 1AB", "Maidstone", "Registered", "Good", "2022-08-09"),
        ("1-100006", "1-000004", "A. Patel Dartford", "DA1 1AB", "Dartford", "Registered", np.nan, np.nan),   # never inspected
        ("1-100007", "1-000003", "Garden of England Swale", "ME10 3AB", "Swale", "Registered", "Inadequate", "2026-01-15"),
        ("1-100008", "1-000002", "Kent Homecare Dover", "CT16 1AB", "Dover", "Deregistered", "Good", "2021-03-03"),
    ]
    cols = ["location_id", "provider_id", "location_name", "postal_code", "local_authority",
            "registration_status", "overall_rating", "overall_rating_date"]
    return pd.DataFrame(rows, columns=cols)


Writing sample_data.py


## Load Weeks 1–2 output (or the labelled sample)

Tries the real files first, using the filenames the Week 1 and Week 2 notebooks actually save. It falls back to the sample so the notebook runs on its own. **Any numbers produced from the sample are illustrative only.**

In [4]:
import importlib, os
import pandas as pd
import load_week3 as lw, sample_data
importlib.reload(lw); importlib.reload(sample_data)

LOCATIONS_PATH = "../Week_1/home_safe_kent_personal_care_locations.csv"
PROVIDERS_PATH = "../Week_2/home_safe_kent_providers_enriched.parquet"

if os.path.exists(LOCATIONS_PATH) and os.path.exists(PROVIDERS_PATH):
    locations = pd.read_csv(LOCATIONS_PATH, dtype=str)   # dtype=str keeps IDs exactly as written
    providers = pd.read_parquet(PROVIDERS_PATH)
    USING_SAMPLE_DATA = False
    print(f"Real inputs: {len(locations)} locations, {len(providers)} providers")
else:
    locations, providers = sample_data.sample_locations(), sample_data.sample_providers()
    USING_SAMPLE_DATA = True
    print("Real Week 1/2 outputs not found -- using the labelled SAMPLE (illustrative numbers only).")
locations

Real Week 1/2 outputs not found -- using the labelled SAMPLE (illustrative numbers only).


,location_id,provider_id,location_name,postal_code,local_authority,registration_status,overall_rating,overall_rating_date
0,1-100001,1-000001,Home Safe Ashford,TN23 1AB,Ashford,Registered,Good,2024-05-14
1,1-100002,1-000001,Home Safe Canterbury,CT1 2AB,Canterbury,Registered,Outstanding,2023-11-02
2,1-100003,1-000002,Kent Homecare Canterbury,CT2 7AB,Canterbury,Registered,Requires improvement,2025-02-20
3,1-100004,1-000002,Kent Homecare Thanet,CT9 1AB,Thanet,Registered,Good,2025-06-30
4,1-100005,1-000003,Garden of England Maidstone,ME14 1AB,Maidstone,Registered,Good,2022-08-09
5,1-100006,1-000004,A. Patel Dartford,DA1 1AB,Dartford,Registered,NaN,NaN
6,1-100007,1-000003,Garden of England Swale,ME10 3AB,Swale,Registered,Inadequate,2026-01-15
7,1-100008,1-000002,Kent Homecare Dover,CT16 1AB,Dover,Deregistered,Good,2021-03-03


## Task 2 — Why idempotency matters: break it first

The tempting Week 2 shortcut, `to_sql(..., if_exists="append")`, run twice. The second run is what happens when a scheduled job retries, or someone re-runs a cell.

In [5]:
naive = sqlite3.connect(":memory:")
for run in (1, 2):
    locations.to_sql("locations_naive", naive, if_exists="append", index=False)
    n = naive.execute("SELECT COUNT(*) FROM locations_naive").fetchone()[0]
    print(f"After naive run {run}: {n} rows")
naive.close()
print("\nSame data, loaded twice -> every provider count in the Week 4 dashboard is now doubled.")

After naive run 1: 8 rows
After naive run 2: 16 rows

Same data, loaded twice -> every provider count in the Week 4 dashboard is now doubled.


## Task 3 — Quality checks as gates

Checks run **before** the database is touched. Two kinds:
- **Blocking**: a wrong answer is worse than a late one. Examples: duplicate IDs, orphan providers, an unknown rating label, a future rating date, more than 5% bad postcodes, a row count more than 20% different from the last snapshot.
- **Warning**: loaded, but flagged in `load_audit`. Examples: a handful of bad postcodes, a local authority outside the 12 Kent districts. This follows the Week 2 rule of flagging problems rather than dropping rows.

Ask the room: *"Stale data or wrong data: which is worse for Home Safe's launch decision?"*

In [6]:
from datetime import date
SNAPSHOT = date(2026, 9, 17)
report = lw.run_quality_checks(locations, providers, SNAPSHOT)
print("OK to load:", report.ok_to_load)
report.to_frame()

OK to load: True


,name,passed,blocking,detail
0,required_columns_present,True,True,missing locations=[] providers=[]
1,locations_not_empty,True,True,8 rows
2,location_and_provider_ids_present,True,True,"blank location_id=0, blank provider_id=0"
3,location_id_unique,True,True,duplicates: []
4,provider_id_unique,True,True,duplicates: []
5,every_location_has_a_provider,True,True,orphan provider_ids: []
6,overall_rating_in_allowed_values,True,True,unexpected: []
7,rating_dates_valid_and_not_future,True,True,"unparseable=0, after snapshot_date=0"
8,postcode_invalid_share_within_threshold,True,True,0 invalid (0.0%); limit 5%
9,all_postcodes_valid,True,False,0 invalid postcode(s) loaded and flagged


## Task 4 — The idempotent load: run it three times

In [7]:
DB_PATH = "home_safe_kent.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)          # demo only -- never delete the real database in a scheduled job

conn = lw.connect(DB_PATH)
for run in (1, 2, 3):
    status, rep = lw.run_pipeline(conn, locations, providers, SNAPSHOT)
    n = conn.execute("SELECT COUNT(*) FROM fact_location_snapshot").fetchone()[0]
    print(f"Run {run}: {status}, fact rows = {n}")

print("\nDimension row counts:")
for t in ["dim_provider", "dim_area", "dim_rating_period"]:
    print(f"  {t}: {conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]}")

Run 1: LOADED, fact rows = 8


Run 2: LOADED, fact rows = 8
Run 3: LOADED, fact rows = 8

Dimension row counts:
  dim_provider: 4
  dim_area: 7
  dim_rating_period: 8


**How it works:** inside **one transaction**: upsert dimensions → `DELETE` this `snapshot_date`'s fact rows → `INSERT` them again → check the counts match. If anything fails part-way, the whole transaction rolls back. A different `snapshot_date` **adds** history instead of replacing it:

In [8]:
lw.run_pipeline(conn, locations, providers, date(2026, 10, 17))
pd.read_sql("SELECT snapshot_date, COUNT(*) AS rows FROM fact_location_snapshot GROUP BY snapshot_date", conn)

,snapshot_date,rows
0,2026-09-17,8
1,2026-10-17,8


## Task 5 — Watch a gate block a bad load

In [9]:
bad = locations.copy()
bad.loc[bad.index[0], "overall_rating"] = "Excellent"                 # not a CQC rating
bad.loc[bad.index[1], "location_id"] = bad.loc[bad.index[0], "location_id"]   # duplicate key

status, rep = lw.run_pipeline(conn, bad, providers, SNAPSHOT)
print("Status:", status)
for f in rep.blocking_failures:
    print(f"  BLOCKED by {f.name}: {f.detail}")
print("Fact rows for", SNAPSHOT, "still:",
      conn.execute("SELECT COUNT(*) FROM fact_location_snapshot WHERE snapshot_date=?", (SNAPSHOT.isoformat(),)).fetchone()[0])

Status: BLOCKED
  BLOCKED by location_id_unique: duplicates: ['1-100001']
  BLOCKED by overall_rating_in_allowed_values: unexpected: ['Excellent']
Fact rows for 2026-09-17 still: 8


In [10]:
pd.read_sql("SELECT snapshot_date, status, rows_in, rows_loaded, blocking_failed, warnings FROM load_audit", conn)

,snapshot_date,status,rows_in,rows_loaded,blocking_failed,warnings
0,2026-09-17,LOADED,8,8,,
1,2026-09-17,LOADED,8,8,,
2,2026-09-17,LOADED,8,8,,
3,2026-10-17,LOADED,8,8,,
4,2026-09-17,BLOCKED,8,0,location_id_unique: duplicates: ['1-100001']; ...,


## Task 6 — Automated tests (pytest)

21 tests, in-memory SQLite, no network, no keys. They cover idempotency, history, dimension upserts, every blocking gate, warnings that don't block, the drift override, **rollback after a simulated crash mid-load**, and the audit trail.

In [11]:
%%writefile test_load_week3.py
"""pytest suite for load_week3.py -- no network, no API keys, in-memory SQLite."""
from datetime import date

import numpy as np
import pytest

import load_week3 as lw
from sample_data import sample_locations, sample_providers

SNAP = date(2026, 9, 17)


@pytest.fixture
def conn():
    c = lw.connect(":memory:")
    yield c
    c.close()


def fact_count(conn, snap=None):
    if snap is None:
        return conn.execute("SELECT COUNT(*) FROM fact_location_snapshot").fetchone()[0]
    return conn.execute("SELECT COUNT(*) FROM fact_location_snapshot WHERE snapshot_date=?",
                        (snap.isoformat(),)).fetchone()[0]


# --- helpers ---------------------------------------------------------------
def test_is_blank_treats_nan_as_blank():
    assert lw.is_blank(np.nan) and lw.is_blank(None) and lw.is_blank("  ")
    assert not lw.is_blank("Good")


def test_rating_period_key_month_grain_and_not_rated():
    assert lw.rating_period_key("Good", "2024-05-14") == "2024-05"
    assert lw.rating_period_key("Good", "2024-05-14T00:00:00") == "2024-05"
    assert lw.rating_period_key(np.nan, np.nan) == lw.NOT_RATED_KEY


# --- happy path & idempotency ---------------------------------------------
def test_clean_sample_loads(conn):
    status, report = lw.run_pipeline(conn, sample_locations(), sample_providers(), SNAP)
    assert status == "LOADED", report.to_frame()
    assert fact_count(conn) == len(sample_locations())


def test_rerun_same_snapshot_is_idempotent(conn):
    for _ in range(3):
        lw.run_pipeline(conn, sample_locations(), sample_providers(), SNAP)
    assert fact_count(conn) == len(sample_locations())
    assert conn.execute("SELECT COUNT(*) FROM dim_provider").fetchone()[0] == len(sample_providers())
    assert conn.execute("SELECT COUNT(*) FROM dim_area").fetchone()[0] == 7


def test_new_snapshot_date_adds_history_not_duplicates(conn):
    lw.run_pipeline(conn, sample_locations(), sample_providers(), SNAP)
    lw.run_pipeline(conn, sample_locations(), sample_providers(), date(2026, 10, 17))
    assert fact_count(conn) == 2 * len(sample_locations())


def test_rerun_with_corrected_rating_replaces_not_appends(conn):
    lw.run_pipeline(conn, sample_locations(), sample_providers(), SNAP)
    locs = sample_locations()
    locs.loc[locs["location_id"] == "1-100003", "overall_rating"] = "Good"
    lw.run_pipeline(conn, locs, sample_providers(), SNAP)
    rows = conn.execute("SELECT overall_rating FROM fact_location_snapshot "
                        "WHERE location_id='1-100003'").fetchall()
    assert rows == [("Good",)]


def test_provider_dimension_upserts_changed_name(conn):
    lw.run_pipeline(conn, sample_locations(), sample_providers(), SNAP)
    provs = sample_providers()
    provs.loc[0, "provider_name"] = "Home Safe Care Group Ltd"
    lw.run_pipeline(conn, sample_locations(), provs, SNAP)
    names = conn.execute("SELECT provider_name FROM dim_provider WHERE provider_id='1-000001'").fetchall()
    assert names == [("Home Safe Care Group Ltd",)]


def test_unrated_location_gets_not_rated_key(conn):
    lw.run_pipeline(conn, sample_locations(), sample_providers(), SNAP)
    row = conn.execute("SELECT rating_period_key, is_rated, overall_rating FROM fact_location_snapshot "
                       "WHERE location_id='1-100006'").fetchone()
    assert row == (lw.NOT_RATED_KEY, 0, None)


# --- gates block and leave the database unchanged --------------------------
def test_duplicate_location_blocks_and_writes_nothing(conn):
    lw.run_pipeline(conn, sample_locations(), sample_providers(), SNAP)
    bad = sample_locations()
    bad.loc[1, "location_id"] = bad.loc[0, "location_id"]
    status, report = lw.run_pipeline(conn, bad, sample_providers(), SNAP)
    assert status == "BLOCKED"
    assert "location_id_unique" in [r.name for r in report.blocking_failures]
    assert fact_count(conn) == len(sample_locations())  # previous good load untouched


def test_duplicate_provider_id_blocks(conn):
    provs = sample_providers()
    provs.loc[1, "provider_id"] = provs.loc[0, "provider_id"]
    status, report = lw.run_pipeline(conn, sample_locations(), provs, SNAP)
    assert status == "BLOCKED" and fact_count(conn) == 0


def test_orphan_provider_blocks(conn):
    locs = sample_locations()
    locs.loc[0, "provider_id"] = "1-999999"
    status, report = lw.run_pipeline(conn, locs, sample_providers(), SNAP)
    assert status == "BLOCKED"
    assert "every_location_has_a_provider" in [r.name for r in report.blocking_failures]


def test_unknown_rating_label_blocks(conn):
    locs = sample_locations()
    locs.loc[0, "overall_rating"] = "Excellent"
    status, _ = lw.run_pipeline(conn, locs, sample_providers(), SNAP)
    assert status == "BLOCKED"


def test_future_rating_date_blocks(conn):
    locs = sample_locations()
    locs.loc[0, "overall_rating_date"] = "2027-01-01"
    status, _ = lw.run_pipeline(conn, locs, sample_providers(), SNAP)
    assert status == "BLOCKED"


def test_missing_column_blocks_cleanly(conn):
    status, report = lw.run_pipeline(conn, sample_locations().drop(columns=["postal_code"]),
                                     sample_providers(), SNAP)
    assert status == "BLOCKED"
    assert report.blocking_failures[0].name == "required_columns_present"


def test_one_bad_postcode_in_many_is_a_warning_not_a_block(conn):
    locs = sample_locations()
    big = locs.loc[locs.index.repeat(3)].reset_index(drop=True)   # 24 rows
    big["location_id"] = [f"1-2{i:05d}" for i in range(len(big))]
    big.loc[0, "postal_code"] = "NOT A POSTCODE"                   # 1/24 = 4.2%
    status, report = lw.run_pipeline(conn, big, sample_providers(), SNAP)
    assert status == "LOADED"
    assert "all_postcodes_valid" in [r.name for r in report.warnings]
    assert conn.execute("SELECT postcode_valid FROM fact_location_snapshot "
                        "WHERE postal_code='NOT A POSTCODE'").fetchone() == (0,)


def test_many_bad_postcodes_block(conn):
    locs = sample_locations()
    locs.loc[:1, "postal_code"] = "XXX"   # 2/8 = 25%
    status, _ = lw.run_pipeline(conn, locs, sample_providers(), SNAP)
    assert status == "BLOCKED"


def test_non_kent_authority_is_flagged_not_dropped(conn):
    locs = sample_locations()
    locs.loc[0, "local_authority"] = "Medway"
    status, report = lw.run_pipeline(conn, locs, sample_providers(), SNAP)
    assert status == "LOADED"
    assert "local_authority_in_kent_list" in [r.name for r in report.warnings]
    assert conn.execute("SELECT is_kent_district FROM dim_area WHERE local_authority='Medway'").fetchone() == (0,)


def test_row_count_drift_blocks_then_override_allows(conn):
    lw.run_pipeline(conn, sample_locations(), sample_providers(), SNAP)
    half = sample_locations().head(4)                       # 50% drop vs last snapshot
    nxt = date(2026, 10, 17)
    status, _ = lw.run_pipeline(conn, half, sample_providers(), nxt)
    assert status == "BLOCKED" and fact_count(conn, nxt) == 0
    status, _ = lw.run_pipeline(conn, half, sample_providers(), nxt, allow_row_count_change=True)
    assert status == "LOADED" and fact_count(conn, nxt) == 4


def test_failure_mid_load_rolls_back(conn, monkeypatch):
    lw.run_pipeline(conn, sample_locations(), sample_providers(), SNAP)

    def boom(*a, **k):
        raise RuntimeError("simulated crash after DELETE")
    real_upsert = lw._upsert_rating_periods

    def crash_after_delete(c, keys):
        real_upsert(c, keys)
        c.execute("DELETE FROM fact_location_snapshot WHERE snapshot_date=?", (SNAP.isoformat(),))
        boom()
    monkeypatch.setattr(lw, "_upsert_rating_periods", crash_after_delete)
    with pytest.raises(RuntimeError):
        lw.run_pipeline(conn, sample_locations(), sample_providers(), SNAP)
    assert fact_count(conn) == len(sample_locations())   # delete was rolled back
    assert conn.execute("SELECT status FROM load_audit ORDER BY rowid DESC LIMIT 1").fetchone() == ("FAILED",)


def test_every_run_is_audited(conn):
    lw.run_pipeline(conn, sample_locations(), sample_providers(), SNAP)
    bad = sample_locations()
    bad.loc[0, "overall_rating"] = "Excellent"
    lw.run_pipeline(conn, bad, sample_providers(), SNAP)
    statuses = [r[0] for r in conn.execute("SELECT status FROM load_audit ORDER BY rowid")]
    assert statuses == ["LOADED", "BLOCKED"]


def test_density_query_runs_and_excludes_deregistered(conn):
    lw.run_pipeline(conn, sample_locations(), sample_providers(), SNAP)
    rows = {r[0]: r for r in conn.execute(lw.DENSITY_BY_AREA_SQL)}
    assert "Dover" not in rows                   # only location there is Deregistered
    assert rows["Canterbury"][2] == 2            # two registered locations
    # Dartford's only location is unrated: counts must be 0, not NULL (NULL-comparison trap)
    assert rows["Dartford"][3:8] == (0, 0, 0, 0, 1)
    assert rows["Dartford"][8] is None           # no rated locations -> share undefined, not 0


Writing test_load_week3.py


In [12]:
!python -m pytest test_load_week3.py -q -p no:cacheprovider --color=no

...

..

..

.......

...

....                                                    [100%]
21 passed in 0.60s


## Task 7 — A week 4 preview: does the schema answer the business question?

Density and rating mix by district, for the latest snapshot and **currently registered** locations only. Deregistered locations stay in the fact table for history but aren't competitors.

In [13]:
pd.read_sql(lw.DENSITY_BY_AREA_SQL, conn)

,local_authority,providers,locations,outstanding,good,requires_improvement,inadequate,not_yet_rated,good_or_better_share
0,Canterbury,2,2,1,0,1,0,0,0.5
1,Thanet,1,1,0,1,0,0,0,1.0
2,Swale,1,1,0,0,0,1,0,0.0
3,Maidstone,1,1,0,1,0,0,0,1.0
4,Dartford,1,1,0,0,0,0,1,NaN
5,Ashford,1,1,0,1,0,0,0,1.0


In [14]:
conn.close()

## Task 8 — Scheduling: a notebook is not a pipeline

A scheduler can't click "Run all". It needs a **script** that runs unattended and returns an **exit code**: 0 = loaded, 2 = blocked by a gate, 1 = crashed. That's how cron, Task Scheduler or GitHub Actions knows a run went wrong.

In [15]:
%%writefile run_pipeline.py
"""Home Safe -- scheduled entry point for the Week 3 load.

This is what cron / Windows Task Scheduler / GitHub Actions runs. A notebook is
for exploring; a scheduler needs a plain script with a meaningful exit code.

Exit codes:
  0  loaded successfully
  2  blocked by a data-quality gate (database unchanged)
  1  unexpected failure (database rolled back)

Example:
  python run_pipeline.py --locations ../Week_1/home_safe_kent_personal_care_locations.csv \
                         --providers ../Week_2/home_safe_kent_providers_enriched.parquet \
                         --db home_safe_kent.db
"""
import argparse
import logging
import sys
from datetime import date
from pathlib import Path

import pandas as pd

import load_week3 as lw


def read_table(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Input not found: {p.resolve()}")
    if p.suffix.lower() == ".parquet":
        return pd.read_parquet(p)  # needs pyarrow
    # dtype=str keeps IDs like '00000006' and '1-000001' exactly as written
    return pd.read_csv(p, dtype=str, keep_default_na=True)


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description="Home Safe Week 3 load")
    ap.add_argument("--locations", required=True, help="Week 1 locations CSV")
    ap.add_argument("--providers", required=True, help="Week 2 enriched providers (.parquet or .csv)")
    ap.add_argument("--db", default="home_safe_kent.db")
    ap.add_argument("--snapshot-date", default=date.today().isoformat(),
                    help="YYYY-MM-DD; defaults to today. Re-using a date REPLACES that snapshot.")
    ap.add_argument("--allow-row-count-change", action="store_true",
                    help="Operator override for the volume-drift gate. Record why in the runbook log.")
    ap.add_argument("--log-file", default="pipeline.log")
    args = ap.parse_args(argv)

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s %(levelname)s %(message)s",
        handlers=[logging.FileHandler(args.log_file), logging.StreamHandler(sys.stdout)],
    )
    log = logging.getLogger("home_safe")

    try:
        snapshot = date.fromisoformat(args.snapshot_date)
        locations = read_table(args.locations)
        providers = read_table(args.providers)
        conn = lw.connect(args.db)
        try:
            status, report = lw.run_pipeline(conn, locations, providers, snapshot,
                                             allow_row_count_change=args.allow_row_count_change)
        finally:
            conn.close()
    except Exception:
        log.exception("Pipeline FAILED -- database transaction rolled back")
        return 1

    for r in report.results:
        level = logging.INFO if r.passed else (logging.ERROR if r.blocking else logging.WARNING)
        log.log(level, "check %-42s %s  %s", r.name, "PASS" if r.passed else "FAIL", r.detail)

    if status == "BLOCKED":
        log.error("Load BLOCKED by data-quality gate for snapshot %s -- nothing written", snapshot)
        return 2
    log.info("Load complete for snapshot %s (%d locations)", snapshot, len(locations))
    return 0


if __name__ == "__main__":
    sys.exit(main())


Writing run_pipeline.py


In [16]:
import subprocess
sample_data.sample_locations().to_csv("sample_locations.csv", index=False)
sample_data.sample_providers().to_csv("sample_providers.csv", index=False)
cmd = [sys.executable, "run_pipeline.py", "--locations", "sample_locations.csv",
       "--providers", "sample_providers.csv", "--db", "scheduled_demo.db",
       "--snapshot-date", "2026-09-17"]
for attempt in (1, 2):
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(f"Attempt {attempt}: exit code {result.returncode}")
print(result.stdout.splitlines()[-1])

Attempt 1: exit code 0


Attempt 2: exit code 0
2026-09-17 18:22:17,707 INFO Load complete for snapshot 2026-09-17 (8 locations)


### Pick the scheduler that matches your machine

**Order matters:** the scheduled job must **extract (Week 1) → transform (Week 2) → load (this script)**. Scheduling only the load re-loads the same old CSV forever. Wrap your three scripts in one shell/batch file, or one Python `main`, and schedule that.

**macOS / Linux: cron (the spec baseline).** Edit with `crontab -e`. This line runs daily at 06:30 local time:
```
30 6 * * * cd /path/to/home_safe && /path/to/venv/bin/python run_pipeline.py --locations ../Week_1/home_safe_kent_personal_care_locations.csv --providers ../Week_2/home_safe_kent_providers_enriched.parquet >> cron.log 2>&1
```
Use absolute paths. Cron doesn't load your shell profile or activate your virtual environment.

**Windows: Task Scheduler.** Most laptops in the room will need this, since Windows has no cron. From Command Prompt:
```
schtasks /create /tn "HomeSafePipeline" /sc daily /st 06:30 /tr "C:\path\to\run_home_safe.bat"
```
Put the `cd` plus `python run_pipeline.py …` lines in `run_home_safe.bat`. The laptop must be on and awake at that time.

**Google Colab:** free Colab has no built-in scheduler, and sessions time out. Use Colab to develop, and one of the other options to schedule.

**GitHub Actions (stretch goal, works from any laptop).** Illustrative, not executed here:
```yaml
name: home-safe-refresh
on:
  schedule:
    - cron: "37 5 * * 1"     # Mondays 05:37 UTC -- off the top of the hour (GitHub: busiest, may delay/drop)
  workflow_dispatch: {}      # manual "Run workflow" button for testing
jobs:
  refresh:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install pandas requests rapidfuzz pyarrow
      - run: python extract.py && python transform.py && python run_pipeline.py --locations data/locations.csv --providers data/providers.parquet
        env:
          CQC_SUBSCRIPTION_KEY: ${{ secrets.CQC_SUBSCRIPTION_KEY }}
          COMPANIES_HOUSE_API_KEY: ${{ secrets.COMPANIES_HOUSE_API_KEY }}
```
Three catches to teach, not hide:
1. **Keys go in repository Secrets**, never in the repo. A committed key in a public repo is a leaked key.
2. **The runner is thrown away after each run.** A `home_safe_kent.db` written there is lost unless you save it: commit it back, upload it as a workflow artifact, or load into BigQuery instead. This is the most common surprise.
3. Check the current major versions of `actions/checkout` / `setup-python` in the GitHub Marketplace before using them. The versions above may be out of date.

## Stretch — incremental refresh with CQC's `/changes` endpoint

A full refresh re-pulls every Kent location. CQC's OpenAPI spec also lists `GET /changes/{organisationType}`, which returns what changed in a time window. A pipeline could fetch only those IDs and re-load only them. **Not tested:** the query parameters and response shape have not been checked against a live call. Read them from `syndication.json` and try it before building on it. Good for teams that finish early. In an incremental design, the `DELETE` + `INSERT` in this notebook would become a per-location upsert.

## Week 3 deliverables checklist

- [ ] **Schema design doc**: table list, grain of each table, keys, and your `dim_rating_period` decision with the reason
- [ ] **Load script**: idempotent (prove it: run it twice and show the row count doesn't change) with data-quality gates and an audit table
- [ ] **Tests**: at minimum idempotency, one blocking gate, and rollback
- [ ] **Scheduled refresh**: cron, Task Scheduler or GitHub Actions, with a screenshot or log of one scheduled run
- [ ] **Runbook** (see the guidance doc template): how to re-run safely, what each exit code means, what to do when a gate blocks
- [ ] OGL attribution still in the README: *Contains Care Quality Commission information licensed under the Open Government Licence v3.0.*

**Sources:** [SQLite UPSERT](https://www.sqlite.org/lang_upsert.html); [SQLite transactions](https://www.sqlite.org/lang_transaction.html); [SQLite foreign keys](https://www.sqlite.org/foreignkeys.html); GitHub Docs, [Events that trigger workflows: schedule](https://docs.github.com/en/actions/reference/workflows-and-actions/events-that-trigger-workflows#schedule); CQC `syndication.json` OpenAPI spec; `HomeSafe_Week1-4_Project_Spec` Week 3 section; Week 1 and Week 2 reference notebooks (column names).